# Structural equation modelling and path analysis: lab notebook

**Statistical Modelling and Forecasting, Multivariate and advanced methods**

Run the cells from top to bottom (Runtime > Run all). NumPy, pandas, SciPy, statsmodels and scikit-learn are already installed in Colab. Read each section, run it, and compare your output with the expected output at the end of the notebook.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # keeps the output tidy

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats

rng = np.random.default_rng(11)

### 1. Simulate X -> M -> Y with a small direct path

In [ ]:
# 1. Simulate X -> M -> Y with a small direct path
n, a, b, cp = 300, 0.6, 0.5, 0.15
X = rng.standard_normal(n)
M = a * X + rng.standard_normal(n)
Y = cp * X + b * M + rng.standard_normal(n)
df = pd.DataFrame({"X": X, "M": M, "Y": Y})
print(df.corr().round(3))

### 2. Path coefficients by regression

In [ ]:
# 2. Path coefficients by regression
f1 = sm.OLS(df.M, sm.add_constant(df[["X"]])).fit()
f2 = sm.OLS(df.Y, sm.add_constant(df[["X", "M"]])).fit()
f3 = sm.OLS(df.Y, sm.add_constant(df[["X"]])).fit()
ah, bh, ch = f1.params["X"], f2.params["M"], f2.params["X"]
print("\na =", round(ah, 3), "| b =", round(bh, 3), "| c' =", round(ch, 3))
print("indirect a*b =", round(ah * bh, 3), "| total c' + ab =", round(ch + ah * bh, 3), "| c from Y~X:", round(f3.params["X"], 3))

### 3. Sobel test of the indirect effect and a bootstrap interval

In [ ]:
# 3. Sobel test of the indirect effect and a bootstrap interval
sa, sb = f1.bse["X"], f2.bse["M"]
z = ah * bh / np.sqrt(bh ** 2 * sa ** 2 + ah ** 2 * sb ** 2)
print("Sobel z =", round(z, 2), "| p =", format(2 * stats.norm.sf(abs(z)), ".2e"))
boots = []
for _ in range(1000):
    i = rng.integers(0, n, n); d = df.iloc[i]
    a_ = sm.OLS(d.M, sm.add_constant(d[["X"]])).fit().params["X"]
    b_ = sm.OLS(d.Y, sm.add_constant(d[["X", "M"]])).fit().params["M"]
    boots.append(a_ * b_)
print("bootstrap 95% interval for ab:", np.round(np.percentile(boots, [2.5, 97.5]), 3))

### 4. Path analysis from correlations only (the numerical problem)

In [ ]:
# 4. Path analysis from correlations only (the numerical problem)
rxm, rmy, rxy = 0.5, 0.5 + 0.2 * 0.5, 0.2 + 0.5 * 0.5
R = np.array([[1, rxm, rxy], [rxm, 1, rmy], [rxy, rmy, 1]])
beta = np.linalg.solve(R[:2, :2], R[:2, 2])
print("\nfrom correlations: b =", round(beta[1], 3), "| c' =", round(beta[0], 3), "| R2 =", round(beta @ R[:2, 2], 3))

### 5. Model fit: chi-square test of the model with no direct path (1 degree of freedom)

In [ ]:
# 5. Model fit: chi-square test of the model with no direct path (1 degree of freedom)
S = np.corrcoef(df.values.T)
Sh = S.copy(); Sh[0, 2] = Sh[2, 0] = S[0, 1] * S[1, 2]
Fml = np.log(np.linalg.det(Sh)) - np.log(np.linalg.det(S)) + np.trace(S @ np.linalg.inv(Sh)) - 3
chi = (n - 1) * Fml
print("implied r(X,Y) without c':", round(Sh[0, 2], 3), "| observed:", round(S[0, 2], 3))
print("chi2(1) =", round(chi, 2), "| p =", round(stats.chi2.sf(chi, 1), 4))

### 6. A real direct path makes the no-direct-path model fail (a true model is still rejected about 5% of the time by chance)

In [ ]:
# 6. A real direct path makes the no-direct-path model fail (a true model is still rejected about 5% of the time by chance)
for cp2 in (0.0, 0.3, 0.6):
    X2 = rng.standard_normal(n); M2 = a * X2 + rng.standard_normal(n); Y2 = cp2 * X2 + b * M2 + rng.standard_normal(n)
    S2 = np.corrcoef(np.column_stack([X2, M2, Y2]).T); S2h = S2.copy(); S2h[0, 2] = S2h[2, 0] = S2[0, 1] * S2[1, 2]
    c2 = (n - 1) * (np.log(np.linalg.det(S2h)) - np.log(np.linalg.det(S2)) + np.trace(S2 @ np.linalg.inv(S2h)) - 3)
    print(f"true direct effect {cp2:.1f}: chi2 = {c2:6.2f}, p = {stats.chi2.sf(c2, 1):.4f}")

## Expected output

Running all cells above should print:

```
       X      M      Y
X  1.000  0.545  0.376
M  0.545  1.000  0.544
Y  0.376  0.544  1.000

a = 0.719 | b = 0.509 | c' = 0.157
indirect a*b = 0.366 | total c' + ab = 0.523 | c from Y~X: 0.523
Sobel z = 6.7 | p = 2.03e-11
bootstrap 95% interval for ab: [0.261 0.498]

from correlations: b = 0.5 | c' = 0.2 | R2 = 0.39
implied r(X,Y) without c': 0.297 | observed: 0.376
chi2(1) = 3.84 | p = 0.0502
true direct effect 0.0: chi2 =   4.33, p = 0.0373
true direct effect 0.3: chi2 =  47.46, p = 0.0000
true direct effect 0.6: chi2 =  48.40, p = 0.0000
```

## Practice exercises

Try these in the cells below.

1. Set the true direct effect to 0 and the indirect path strong. Compare the χ² of the model that drops the x → y path.
2. Set a strong direct path, then constrain it to zero. What happens to the fit?
3. Reduce n and watch the Sobel z for the indirect effect.

In [ ]:
# Exercise 1


In [ ]:
# Exercise 2


In [ ]:
# Exercise 3
